# **Week 8 Examples - Advanced Function and Class Mechanics**

By [**Binghuan Li**](mailto:binghuan.li19@imperial.ac.uk), November 2025

---

❗ **A few notes ahead**: Welcome to week 8 of Programming 2! 🌞 In this notebook, we shall delve into the world of **function decorators** in Python. By means of decorator, we add a line above the function definition, *e.g.*,

```Python
@staticmethod
def myFunction():
  pass
```

In this example, we say `myFunction` is decorated by `@staticmethod`.

You may ask, "what does function decorators do❓". Well, in short, a decorator modifies or extends the behaviour a function without changing the function's code. However, different decorators may introduce distinct behaviours, and they are highly customizable.

Confused? 🤔 At least I am. Let us look at a few examples, so and hopefully, they will clear out our frustrations...Especially, we shall explore the usage of function decorators in three dimensions:

1. class methods,

2. static methods,

3. wrapper functions.
---

🦮 **Guide**: This is a bit lengthy notebook that compiles a few distinct topics. You can use the fold function (the `>` button next to the titles) to hide undesired cells for a better focus.

## Class Methods



### Class Methods as An Object Constructer

Possibly, the most common and straightforward usage of a class method is instantiating an object by providing *an alternative* constructor.

Unlike the conventional ways to instantiate a new object by *directly* calling the class constructor (*i.e.*, sending arguments to `__init__`), class methods use the keyword `cls`, which operates on the classes' level, to revoke the `__init__` *after* processing the data required to instantiate the object.

Explore this example! 🙂

In [ ]:
from datetime import date

class Person:

  def __init__(self, name: str, age: int):
      self.name = name;
      self.age = age;

  @classmethod    # <-- method decorator, must be @classmethod
  def fromBirthYear(cls, name, birth_year):
    """
    Comments to follow:
    ===================
    1. Instread of supplying the keyword 'self', 'cls' in the function
    argument refers to the class itself.

    2. cls(name, age) below revokes the __init__ method to construct the
    class instance.

    3. Hence we say, 'self' operates on the instances' level, 'cls'
    operates on the classes' level. See below.
    """
    age = date.today().year - birth_year;
    return cls(name, age);


# test cases
p1 = Person('Meier', 20);
print(f'{p1.name} is of age {p1.age}');

p2 = Person.fromBirthYear('Meyer', 2004);
print(f'{p2.name} is of age {p2.age}');


Meier is of age 20
Meyer is of age 20


### Class Methods Modify the Class States

The term *class states* refers to the *class attributes*, which are declared in the class body (outside of any method) in a class definition. These attributes are usually placed near the top of the class definition but outside of any instance or class method.

To excerpt this further, we first need to differentiate the concepts of **class attribute** and **instance attribute**:

- **Instance attributes** are identified and accessed (commonly) with the word `self` (*e.g.*, `self.name`, `self.age`). They are only defined within the methods, not outside of the methods. ***Instance attributes are the 'attributes' we referred to, during your Programming 2 lectures and labs.*** 🤝

- **Class attributes** are commonly defined outside of any methods, and accessed via an implicit name `cls` (*e.g.*, `cls.count`, see below). They are shared by *all instances*, serving as the 'global variables' in a class. Modifying a class attribute will affect all instances of the class. ***We have barely seen or used class attributes, thus far.***

Confused? At least I am. 🤦 Let us look at an example together.

In [ ]:
class Person:
    count = 0;   # <---- this is a class attribute!

    def __init__(self, name: str):
        self.name = name;    # <---- this is an instance attribute!
        Person.count += 1;   # a 'global' counter that tracks of No. instances

    @classmethod
    def get_population(cls):
        """Returns the current population (number of instances created)"""
        return f"Population: {cls.count}";

    @classmethod
    def reset_population(cls):
        """Resets the global population counter to zero."""
        cls.count = 0

# test cases
student = Person("Webster")
professor = Person("Prof. Weinberg")
tutor = Person("Mr. Holloway")

print(Person.get_population())
Person.reset_population()   # reset instance count
print(Person.get_population())

Population: 3
Population: 0


## Property Functions

### Your Attributes Are Not Always Safe

Suppose you are working on a database with critical information inside. You want to protect the information to the 'read-only' users from being modified. However, users can always access to the variable, by simply calling *e.g.*, `self.name`, and assign a new value to the variable. 😢

For example...

In [ ]:
class Person:

  def __init__(self, name: str, age: int):
      self.name = name;
      self.age = age;

# test cases
p1 = Person('Peter', 19);
print(f'{p1.name} is of age {p1.age}');

# If I mofidy the name...
p1.name = 'Pete';   # your attributes are not always safe!
print(f'{p1.name} is of age {p1.age}');

Peter is of age 19
Pete is of age 19


### Getter and Setter Methods

The example shown above is absolutely not ideal! Actually, it is indeed a bad example of **encapsulation** - the code exposes too much information of the internal state of an object (which should be hidden) to the user. Users therefore gained excessive controlled access to the attributes!

You, as a brillant software engineer with an impressive bioengineering background, came up with a method: Instead of letting users directly setting or modifying the values of an attribute by calling `self.<attribute_name>`, you introduce a **setter** method, which can be used to set/update the value, in accompany with a **getter** method to retrieve the value.

🔐 Sounds good! Now, you can provide the 'read-only' users with only a getter but no setter to protect the information!

---

In Python OOP,
- a setter method is decorated by a tag `@property`,
- a getter method is docorated by a tag `<property_name>.setter`, where `<property_name>` is the name of the getter method.

Let us look at the followng example:

#### **Example 1: only getter, no setter**

You only provide the getter to the 'read-only' users, and hides the attribute to protect the `name` from being modified! ❌

In [ ]:
class Person:
    def __init__(self, name: str, age: int):
        self._name = name   # This atrribute is declared as '_name',
                            # in order to distinguish from the method 'name'
                            # defined below. It is *just* a normal attribute -
                            # not a *special* method!
        self.age = age

    @property # <-- getter decorator, must be @property
    def name(self):
        """Getter for the name attribute."""
        return self._name

# test cases
p1 = Person('Peter', 19);
print(p1.name) # this triggers the getter method

Peter


If I try to modify the value...

In [ ]:
p1.name = 'Pete'; # error! no setter provided, and _name is hidden!

NameError: name 'p1' is not defined

#### **Example 2: with both the getter and setter**

You can provide the getter and setter to the 'admin' users, allowing them to modify and get the name info at the same time. ✅

In [ ]:
class Person:
    def __init__(self, name: str, age: int):
        self._name = name   # This atrribute is declared as '_name',
                            # in order to distinguish from the method 'name'
                            # defined below. It is *just* a normal attribute -
                            # not a *special* method!
        self.age = age

    @property # <-- getter decorator, must be @property
    def name(self):
        """Getter for the name attribute."""
        return self._name

    @name.setter  # <-- setter decorator, must be @<property_name>.setter
    def name(self, value):
        """Setter for the name attribute."""
        if not isinstance(value, str):
            raise ValueError("Name must be a string.")
        self._name = value # set the self.name


# test cases
p1 = Person('Peter', 19);
print(p1.name)    # this triggers the *getter* method

p1.name = 'Pete'; # this triggers the *setter* method
print(p1.name)    # this triggers the *getter* method

Peter
Pete


## Wrapper Function

### Don't Repeat Yourself (DRY)

One important principle in software design is [**don't repeat yourself** (DRY)](https://en.wikipedia.org/wiki/Don%27t_repeat_yourself), which promotes the idea of code reusability by avoiding any unnecessary duplications. For example, using functions, classes, and inheritance in OOP are the actions of DRY.

Suppose we want to know the time elapsed for executing a function...

In [ ]:
from math import sqrt
import time

def pythagoras(a: float, b: float):
    c = sqrt(a**2 + b**2);
    return c

def euclidean(a: float, b: float, c: float):
    distance = sqrt(a**2 + b**2 + c**2);
    return distance

t0 = time.time();
c = pythagoras(3, 4);
dt = time.time() - t0;
print(f'function returned the value {c}, time elapsed {dt} s');

t0 = time.time();
distance = euclidean(5, 12, 13);
dt = time.time() - t0;
print(f'function returned the value {distance}, time elapsed {dt} s');

function returned the value 5.0, time elapsed 5.5789947509765625e-05 s
function returned the value 18.384776310850235, time elapsed 6.67572021484375e-05 s


You have seen that the following lines are repeated everytime to time a function:
```
t0 = time.time();
dt = time.time() - t0;
```
⛔ Not ideal!

### The Wrapper

To simplify the timing procedure, we can introduce the **wrapper function**, which give us greater flexibility to pre-process and post-process the data, before and after the main function execution.

Consider the example:

In [ ]:
import time

def debug_timer(some_function):

    def wrapper_function(*args, **kwargs):
        t0 = time.time()
        output = some_function(*args, **kwargs)
        dt = time.time() - t0
        print(f'Elapsed time: {dt} seconds')
        return output

    return wrapper_function

@debug_timer
def pythagoras(a: float, b: float):
    c = sqrt(a**2 + b**2);
    return c

@debug_timer
def euclidean(a: float, b: float, c: float):
    distance = sqrt(a**2 + b**2 + c**2);
    return distance

c = pythagoras(3, 4);
print(c);

distance = euclidean(5, 12, 13);
print(distance);

Elapsed time: 4.291534423828125e-06 seconds
5.0
Elapsed time: 5.7220458984375e-06 seconds
18.384776310850235


In the above example, functions `pythagoras` and `euclidean` are decorated by `debug_timer` (`@debug_timer`), and the function `debug_timer` has a nested function `wrapper_function` defined in the local scope.

❓ What happended when the function `pythagoras` is triggered? See the flow below:

1. `pythagoras` is triggered with two aruguments: `3` and `4`.

2. Since `pythagoras` is decorated by `debug_timer`, instread of going to the line `c = sqrt(a**2 + b**2)` directly, the function `debug_timer` is triggered, with the argument `some_function` being the function handle `pythagoras` (*i.e.* you pass a function into another function).

3. The inner function `wrapper_function` is triggered when the `return wrapper_function` statement is revoked. Then the timing starts, `some_function` (being `pythagoras` here) is executed (`c = sqrt(a**2 + b**2)`), and returns the output (`c`).


### 🧠 Why decorators are useful?

They allow you to:

- Measure performance (time your function)
- Add logging
- Check permissions
- Validate inputs
- *etc.*